In [2]:
from pathlib import Path
import pickle
import numpy as np


In [3]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FAISS_DIR = PROJECT_ROOT / "data" / "faiss_index"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma_db"

FAISS_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)


In [4]:
import pickle
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


c:\Users\shravan\Documents\OMNI_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
with open("../data/processed/chunks.pkl", "rb") as f:
    nodes = pickle.load(f)

print("Loaded chunks:", len(nodes))


Loaded chunks: 23


In [6]:
assert len(nodes) > 0
assert hasattr(nodes[0], "text")
assert hasattr(nodes[0], "metadata")


In [7]:
model = SentenceTransformer("BAAI/bge-base-en")
EMBED_DIM = model.get_sentence_embedding_dimension()

print("Embedding dimension:", EMBED_DIM)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 500.04it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-base-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 768


In [8]:
texts = [node.text for node in nodes]

assert all(isinstance(t, str) and len(t.strip()) > 0 for t in texts)
print("Texts ready for embedding:", len(texts))


Texts ready for embedding: 23


In [9]:
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.array(embeddings).astype("float32")

print("Embeddings shape:", embeddings.shape)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:10<00:00, 10.78s/it]

Embeddings shape: (23, 768)


In [10]:
assert embeddings.shape[0] == len(texts)
assert embeddings.shape[1] == EMBED_DIM


In [11]:
index = faiss.IndexFlatIP(EMBED_DIM)
index.add(embeddings)

print("Total vectors in FAISS:", index.ntotal)


Total vectors in FAISS: 23


In [15]:
faiss.write_index(index, "../data/faiss_index/faiss.index")

with open("../data/faiss_index/faiss_chunks_meta.pkl", "wb") as f:
    pickle.dump(nodes, f)

print("FAISS index and metadata saved")


FAISS index and metadata saved


In [13]:
def search(query: str, k: int = 5):
    q_emb = model.encode(
        query,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        np.array([q_emb]),
        k
    )

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "text": nodes[idx].text[:500],
            "metadata": nodes[idx].metadata
        })

    return results


In [14]:
results = search("difference between synchronous and asynchronous ")

for r in results:
    print("=" * 80)
    print("Score:", r["score"])
    print(r["text"])


Score: 0.8845745325088501
except ZeroDivisionError:
  print('Cannot divide by zero')
24. Explain the difference between synchronous and asynchronous programming in Python. Answer: Synchronous code runs line-by-line; asynchronous uses 'async/await' for concurrent I/O
tasks, improving performance without threading. 25. What are Python's data classes and where would you use them? Answer: Data classes reduce boilerplate for classes that primarily store data. Use '@dataclass'
from the 'dataclasses' module.
Score: 0.7493261694908142
Python Interview Q&A for Data Science and Gen AI
Generated on: June 25, 2025
Experience Level: 2+ Years
1. What are Python's key features that make it suitable for data science and Gen AI development? Answer: Python has simple syntax, a large ecosystem of libraries (like Pandas, NumPy,
Scikit-learn, TensorFlow, PyTorch, Transformers), strong community support, and flexibility for rapid
prototyping. 2. Explain the difference between a list, tuple, set, and diction